In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, r2_score,root_mean_squared_error)
from pathlib import Path
import sys
sys.path.append('..')
from src.preparacao import carregar_dados

caminho = Path('../data/TESTE_ML_COM_TAXAS_INFRA.csv')

df = carregar_dados(caminho)

display(df)

,cod_municipio,nome_municipio,perc_doc_sobrecarga,taxa_informal,taxa_climatizacao,taxa_internet,taxa_lab,media_alunos_por_sala,renda_pc,evasao,tx_repet
0,4201273,Arabutã,34.6,41.70,100.00,100.00,71.43,15.1,2056.91,0.0,0.9
1,4201703,Ascurra,41.5,18.45,63.64,54.55,18.18,29.6,1713.62,3.9,4.8
2,4201950,Balneário Arroio do Silva,35.8,44.02,87.50,75.00,37.50,23.3,1530.60,3.0,2.3
3,4202578,Bom Jesus do Oeste,31.6,41.36,75.00,75.00,50.00,8.7,2347.75,1.8,1.8
4,4203154,Calmon,17.4,30.26,62.50,75.00,25.00,22.2,1031.92,6.1,1.5
...,...,...,...,...,...,...,...,...,...,...,...
290,4216800,São José do Cerrito,25.5,58.87,24.14,65.52,10.34,23.9,1223.88,0.3,3.0
291,4217303,Saudades,26.9,34.33,75.00,81.25,31.25,17.6,2154.73,3.5,7.9
292,4218806,Turvo,33.3,34.14,100.00,93.33,0.00,24.3,2009.39,2.6,4.5
293,4219002,Urussanga,25.6,24.52,89.47,73.68,21.05,21.5,1838.88,2.2,6.6


In [ ]:
from matplotlib.pyplot import grid


colunas_ml = ['perc_doc_sobrecarga','taxa_informal','media_alunos_por_sala','tx_repet','renda_pc',
              'taxa_climatizacao', 'taxa_internet', 'taxa_lab']
x = df[colunas_ml]
y = df['evasao']

x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size=0.2, random_state=42)

modelo = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'max_features': [None, 'sqrt'],      
    'min_samples_leaf': [2, 4, 6],
    'min_samples_split': [4, 8, 12],
    'max_samples': [None, 0.7]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=modelo,
    param_grid=param_grid,
    cv=cv,                        
    scoring='neg_mean_absolute_error',                 
    n_jobs=-1,                    
    verbose=1                     
)

grid_search.fit(x_treino, y_treino)

print("Melhores Hiperparâmetros encontradas:")
print(grid_search.best_params_)

print(f"\nMelhor pontuação média (MAE na validação cruzada): {grid_search.best_score_:.4f}")

df_resultados = pd.DataFrame(grid_search.cv_results_)

colunas_parametros = [col for col in df_resultados.columns if col.startswith('param_')]

colunas_finais = colunas_parametros + ['mean_test_score', 'rank_test_score']

display(df_resultados[colunas_finais].sort_values(by='rank_test_score'))

df_top10 = df_resultados[colunas_finais].sort_values('rank_test_score').head(10)
display(df_top10)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
Melhores Hiperparâmetros encontradas:
{'max_depth': 3, 'max_features': None, 'max_samples': 0.7, 'min_samples_leaf': 6, 'min_samples_split': 4, 'n_estimators': 100}

Melhor pontuação média (MAE na validação cruzada): -1.5140


,param_max_depth,param_max_features,param_max_samples,param_min_samples_leaf,param_min_samples_split,param_n_estimators,mean_test_score,rank_test_score
48,3,None,0.7,6,8,100,-1.514026,1
51,3,None,0.7,6,12,100,-1.514026,1
45,3,None,0.7,6,4,100,-1.514026,1
42,3,None,0.7,4,12,100,-1.515887,4
46,3,None,0.7,6,4,200,-1.517973,5
...,...,...,...,...,...,...,...,...
335,10,None,None,4,4,300,-1.599165,427
218,7,None,None,2,4,300,-1.599281,429
324,10,None,None,2,4,100,-1.599691,430
329,10,None,None,2,8,300,-1.603054,431


,param_max_depth,param_max_features,param_max_samples,param_min_samples_leaf,param_min_samples_split,param_n_estimators,mean_test_score,rank_test_score
48,3,None,0.7,6,8,100,-1.514026,1
51,3,None,0.7,6,12,100,-1.514026,1
45,3,None,0.7,6,4,100,-1.514026,1
42,3,None,0.7,4,12,100,-1.515887,4
46,3,None,0.7,6,4,200,-1.517973,5
52,3,None,0.7,6,12,200,-1.517973,5
49,3,None,0.7,6,8,200,-1.517973,5
36,3,None,0.7,4,4,100,-1.518243,8
39,3,None,0.7,4,8,100,-1.518243,8
43,3,None,0.7,4,12,200,-1.522085,10


In [ ]:
melhor_modelo = grid_search.best_estimator_

melhor_modelo.fit(x_treino, y_treino)

previsoes = melhor_modelo.predict(x_teste)

mae = mean_absolute_error(y_teste, previsoes)
r2 = r2_score(y_teste, previsoes)
rmse = root_mean_squared_error(y_teste, previsoes)

print(f"\nMAE final: {mae:.4f}")
print(f"\nR2 final: {r2:.4f}")
print(f"\nRMSE final: {rmse:.4f}")


MAE final: 1.5743

R2 final: 0.0540

RMSE final: 2.0822


In [4]:
from sklearn.inspection import permutation_importance

melhor_modelo = grid_search.best_estimator_

resultado = permutation_importance(
    melhor_modelo, x_teste, y_teste, n_repeats=10, random_state=42
)

importancia_perm = pd.Series(
    resultado.importances_mean, index=x_teste.columns
).sort_values(ascending=False)
print(importancia_perm)

tx_repet                 0.096681
renda_pc                 0.018473
perc_doc_sobrecarga      0.017210
taxa_lab                 0.009156
media_alunos_por_sala    0.003414
taxa_internet           -0.000490
taxa_informal           -0.000799
taxa_climatizacao       -0.002107
dtype: float64
